# Sesión 4: Consultas Básicas con SELECT
## Clase 4 - Fundamentos de Bases de Datos

En esta sesión aprenderemos:
- SELECT: Selección de columnas específicas
- FROM: Selección de tablas
- WHERE: Filtrado de resultados
- Operadores de comparación, lógicos y BETWEEN
- IN y LIKE para búsquedas avanzadas

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

# Crear conexión SQLite en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✅ Conexión SQLite establecida para Sesión 4")

## SETUP: Crear tabla de envíos (Slide 33)

Usaremos una tabla `envios` para todas las consultas de esta sesión.

In [ ]:
# Crear tabla envios (Slide 33)
cursor.execute('''
    CREATE TABLE envios (
        id_envio INTEGER PRIMARY KEY,
        ciudad_destino TEXT NOT NULL,
        estado TEXT NOT NULL,
        fecha_envio DATE NOT NULL,
        peso_kg NUMERIC,
        cliente_nombre TEXT,
        tipo_servicio TEXT
    )
''')

# Insertar datos de ejemplo
datos_envios = [
    (1, 'Valparaíso', 'En tránsito', '2024-07-01', 5.5, 'Ana García', 'Express'),
    (2, 'Santiago', 'Entregado', '2024-06-20', 2.3, 'Carlos López', 'Estándar'),
    (3, 'Iquique', 'En tránsito', '2024-07-10', 8.2, 'María Rodríguez', 'Express'),
    (4, 'Valparaíso', 'Cancelado', '2024-07-05', 3.1, 'Pedro Martínez', 'Estándar'),
    (5, 'Arica', 'En tránsito', '2024-06-29', 1.8, 'Elena Fernández', 'Express'),
    (6, 'Concepción', 'Entregado', '2024-06-15', 4.5, 'Juan Silva', 'Estándar'),
    (7, 'Santiago', 'En tránsito', '2024-07-08', 6.7, 'Laura Díaz', 'Express'),
    (8, 'Iquique', 'Pendiente', '2024-07-12', 2.2, 'Roberto Gómez', 'Estándar'),
    (9, 'Valparaíso', 'En tránsito', '2024-07-09', 5.0, 'Sofía Torres', 'Express'),
    (10, 'Arica', 'Entregado', '2024-06-25', 3.3, 'Marco Sánchez', 'Estándar')
]

cursor.executemany(
    'INSERT INTO envios VALUES (?, ?, ?, ?, ?, ?, ?)',
    datos_envios
)
conn.commit()

print("✅ Tabla 'envios' creada con 10 registros de ejemplo")

## 1. SELECT: Selección de Columnas (Slides 4-11)

### Problema: ¿Cómo consultar solo la información específica?

La instrucción SELECT permite especificar exactamente qué columnas queremos recuperar, evitando cargar datos innecesarios.

### Sintaxis básica de SELECT (Slide 5)

```sql
SELECT 
    columna1, 
    columna2
FROM nombre_tabla;
```

In [ ]:
# Slide 5: Seleccionar columnas específicas
df = pd.read_sql_query(
    'SELECT id_envio, ciudad_destino, estado FROM envios',
    conn
)

print("Slide 5: SELECT con columnas específicas")
print(df)

### SELECT * - Todas las columnas (Slide 6)

⚠️ **NO se recomienda en producción** por rendimiento y seguridad.

In [ ]:
# Slide 6: Seleccionar TODAS las columnas (útil solo para exploración)
df_todas = pd.read_sql_query(
    'SELECT * FROM envios LIMIT 3',
    conn
)

print("Slide 6: SELECT * (muestra todas las columnas)")
print(f"Columnas: {list(df_todas.columns)}")
print("\nPrimeros 3 registros:")
print(df_todas)

### Alias de columnas (Slide 8)

Renombrar columnas para mejorar claridad en reportes.

In [ ]:
# Slide 8: Usar alias AS para renombrar columnas
df_alias = pd.read_sql_query(
    '''SELECT 
        id_envio AS id,
        ciudad_destino AS ciudad, 
        tipo_servicio AS servicio
    FROM envios''',
    conn
)

print("Slide 8: Alias con AS (renombra columnas para reportes)")
print(df_alias.head())

### Columnas derivadas (Slide 9)

Transformar datos directamente en la consulta.

In [ ]:
# Slide 9: Columnas calculadas o transformadas
df_derivadas = pd.read_sql_query(
    '''SELECT 
        id_envio,
        UPPER(ciudad_destino) AS ciudad_mayuscula,
        UPPER(estado) AS estado_normalizado,
        peso_kg,
        peso_kg * 1000 AS peso_gramos
    FROM envios
    LIMIT 5''',
    conn
)

print("Slide 9: Columnas derivadas (transformaciones)")
print(df_derivadas)

## 2. FROM: Selección de Tabla (Slides 12-20)

### Problema: ¿Qué tabla es la fuente correcta?

La cláusula FROM especifica desde dónde extraer los datos.

### Sintaxis básica de FROM (Slide 13)

```sql
SELECT columna1, columna2
FROM nombre_tabla;
```

In [ ]:
# Slide 13 + 17: Exploración básica con LIMIT
df_exploratorio = pd.read_sql_query(
    'SELECT * FROM envios LIMIT 10',
    conn
)

print("Slide 13 + 17: Consulta exploratoria básica FROM envios")
print(f"\nTotal de registros en tabla: {len(df_exploratorio)}")
print(f"Estructura: {df_exploratorio.shape[0]} filas × {df_exploratorio.shape[1]} columnas")
print("\nTipos de datos:")
print(df_exploratorio.dtypes)

### Alias para tablas (Slide 16)

Cuando hay múltiples tablas, los alias mejoran legibilidad (especialmente con JOINs).

In [ ]:
# Slide 16: Alias de tabla (se usa más con JOINs, aquí es ilustrativo)
df_alias_tabla = pd.read_sql_query(
    '''SELECT 
        e.id_envio,
        e.ciudad_destino,
        e.estado
    FROM envios AS e
    LIMIT 5''',
    conn
)

print("Slide 16: Alias de tabla (e = envios)")
print(df_alias_tabla)

## 3. WHERE: Filtrado de Resultados (Slides 21-28)

### Problema: ¿Cómo restringir resultados a solo lo que necesitamos?

WHERE permite aplicar condiciones para filtrar registros.

### Sintaxis básica WHERE (Slide 22)

In [ ]:
# Slide 22: WHERE con igualdad
df_where_simple = pd.read_sql_query(
    """SELECT 
        id_envio, 
        ciudad_destino,
        estado
    FROM envios
    WHERE estado = 'En tránsito'""",
    conn
)

print("Slide 22: WHERE con condición simple")
print(f"\nEnvíos en tránsito: {len(df_where_simple)}")
print(df_where_simple)

### Operadores de comparación (Slide 23)

In [ ]:
# Slide 23: Operadores de comparación
print("SLIDE 23: Operadores de Comparación")
print("="*70)

# = Igual a
df_igual = pd.read_sql_query(
    "SELECT id_envio, estado FROM envios WHERE estado = 'Entregado' LIMIT 3",
    conn
)
print(f"\n1. Igual a (=): estado = 'Entregado'")
print(df_igual.to_string(index=False))

# <> Distinto de
df_distinto = pd.read_sql_query(
    "SELECT id_envio, estado FROM envios WHERE estado <> 'Cancelado' LIMIT 3",
    conn
)
print(f"\n2. Distinto de (<>): estado <> 'Cancelado'")
print(df_distinto.to_string(index=False))

# > Mayor que
df_mayor = pd.read_sql_query(
    "SELECT id_envio, peso_kg FROM envios WHERE peso_kg > 5 LIMIT 3",
    conn
)
print(f"\n3. Mayor que (>): peso_kg > 5")
print(df_mayor.to_string(index=False))

# <= Menor o igual
df_menor_igual = pd.read_sql_query(
    "SELECT id_envio, peso_kg FROM envios WHERE peso_kg <= 3 LIMIT 3",
    conn
)
print(f"\n4. Menor o igual (<=): peso_kg <= 3")
print(df_menor_igual.to_string(index=False))

### AND, OR y NOT (Slide 24)

Combinar múltiples condiciones.

In [ ]:
# Slide 24: Combinación de condiciones
print("SLIDE 24: Combinación de Condiciones Lógicas")
print("="*70)

# AND: Ambas condiciones deben cumplirse
df_and = pd.read_sql_query(
    """SELECT id_envio, ciudad_destino, estado, fecha_envio
    FROM envios
    WHERE estado = 'En tránsito' AND ciudad_destino = 'Valparaíso'""",
    conn
)
print("\n1. AND (ambas condiciones):")
print("   WHERE estado = 'En tránsito' AND ciudad_destino = 'Valparaíso'")
print(df_and.to_string(index=False))

# OR: Una u otra condición
df_or = pd.read_sql_query(
    """SELECT id_envio, ciudad_destino, estado
    FROM envios
    WHERE ciudad_destino = 'Iquique' OR ciudad_destino = 'Arica'
    LIMIT 5""",
    conn
)
print("\n2. OR (una u otra):")
print("   WHERE ciudad_destino = 'Iquique' OR ciudad_destino = 'Arica'")
print(df_or.to_string(index=False))

# NOT: Excluir
df_not = pd.read_sql_query(
    """SELECT id_envio, estado
    FROM envios
    WHERE estado != 'Cancelado'
    LIMIT 5""",
    conn
)
print("\n3. NOT / != (excluir):")
print("   WHERE estado != 'Cancelado'")
print(df_not.to_string(index=False))

### BETWEEN: Rangos (Slide 25)

In [ ]:
# Slide 25: BETWEEN para rangos
df_between_fecha = pd.read_sql_query(
    """SELECT id_envio, fecha_envio, ciudad_destino
    FROM envios
    WHERE fecha_envio BETWEEN '2024-07-01' AND '2024-07-12'""",
    conn
)

print("SLIDE 25: BETWEEN para rangos")
print("\nEnvíos entre 2024-07-01 y 2024-07-12:")
print(df_between_fecha.to_string(index=False))

# BETWEEN con números
df_between_peso = pd.read_sql_query(
    """SELECT id_envio, peso_kg
    FROM envios
    WHERE peso_kg BETWEEN 2 AND 5""",
    conn
)

print("\nEnvíos con peso entre 2kg y 5kg:")
print(df_between_peso.to_string(index=False))

### IN: Listas de valores (Slide 26)

In [ ]:
# Slide 26: IN para múltiples valores
df_in = pd.read_sql_query(
    """SELECT id_envio, ciudad_destino, estado
    FROM envios
    WHERE ciudad_destino IN ('Santiago', 'Valparaíso', 'Concepción')""",
    conn
)

print("SLIDE 26: IN para listas de valores")
print("\nEnvíos a Santiago, Valparaíso o Concepción:")
print(df_in.to_string(index=False))

print(f"\nTotal: {len(df_in)} envíos")

### LIKE: Búsqueda de texto parcial (Slide 27)

In [ ]:
# Slide 27: LIKE para búsquedas parciales
print("SLIDE 27: LIKE para búsquedas de texto parcial")
print("="*70)

# Comienza con
df_like_comienza = pd.read_sql_query(
    """SELECT id_envio, cliente_nombre
    FROM envios
    WHERE cliente_nombre LIKE 'Mar%'""",
    conn
)
print("\n1. Nombres que COMIENZAN con 'Mar' (%):",)
print(df_like_comienza.to_string(index=False))

# Contiene
df_like_contiene = pd.read_sql_query(
    """SELECT id_envio, cliente_nombre
    FROM envios
    WHERE cliente_nombre LIKE '%a%'""",
    conn
)
print("\n2. Nombres que CONTIENEN 'a' (%):")
print(df_like_contiene.to_string(index=False))

# Un carácter específico
df_like_caracter = pd.read_sql_query(
    """SELECT id_envio, cliente_nombre
    FROM envios
    WHERE cliente_nombre LIKE 'Mar_a%'""",
    conn
)
print("\n3. Nombres que siguen patrón 'Mar_a%' (_):")
if len(df_like_caracter) > 0:
    print(df_like_caracter.to_string(index=False))
else:
    print("   (Sin coincidencias)")

## 4. CASO PRÁCTICO: Reporte Logístico (Slide 34)

Desafío: Obtener envíos en tránsito hacia ciudades específicas del último mes.

In [ ]:
# Slide 34: Caso práctico - Consulta combinada
df_reporte = pd.read_sql_query(
    """SELECT 
        id_envio, 
        cliente_nombre,
        ciudad_destino, 
        estado, 
        fecha_envio,
        tipo_servicio
    FROM envios
    WHERE estado = 'En tránsito' 
      AND ciudad_destino IN ('Valparaíso', 'Iquique', 'Arica')
      AND fecha_envio >= '2024-07-01'""",
    conn
)

print("SLIDE 34: Caso Práctico - Reporte Logístico")
print("="*70)
print("\nQuery: Envíos EN TRÁNSITO a ciudades específicas del último mes")
print("\nCondiciones aplicadas:")
print("  • estado = 'En tránsito'")
print("  • ciudad_destino IN ('Valparaíso', 'Iquique', 'Arica')")
print("  • fecha_envio >= '2024-07-01'")
print(f"\nResultado: {len(df_reporte)} envíos encontrados\n")
print(df_reporte.to_string(index=False))

## 5. TABLA RESUMEN: Métodos de Filtrado (Slide 27)

In [ ]:
# Slide 27: Comparativa de métodos
comparativa = pd.DataFrame({
    'Técnica': ['Igualdad', 'Rango', 'Lista (IN)', 'Texto parcial (LIKE)', 'Combinación lógica'],
    'Sintaxis ejemplo': [
        "WHERE estado = 'Pendiente'",
        "WHERE total_venta BETWEEN 1000 AND 5000",
        "WHERE ciudad IN ('Arica', 'Iquique')",
        "WHERE nombre LIKE 'Ana%'",
        "WHERE estado = 'En tránsito' AND ciudad = 'Valparaíso'"
    ],
    'Uso recomendado': [
        'Filtrar por valor exacto',
        'Valores dentro de rango',
        'Varios valores específicos',
        'Búsquedas parciales',
        'Condiciones múltiples'
    ]
})

print("SLIDE 27: Comparativa de Métodos de Filtrado")
print("\n" + comparativa.to_string(index=False))

## 6. BUENAS PRÁCTICAS Y ERRORES COMUNES (Slides 7, 15, 28)

In [ ]:
buenas_practicas = """
    BUENAS PRÁCTICAS (Slides 7, 15, 28)
    ═════════════════════════════════════════════════════════════
    
    SELECCIÓN DE COLUMNAS (SELECT):
    ✓ Usar SELECT * solo en exploración, nunca en producción
    ✓ Declarar explícitamente columnas necesarias
    ✓ Usar alias para mejorar legibilidad en reportes
    ✓ Aplicar transformaciones solo cuando sea necesario
    
    SELECCIÓN DE TABLA (FROM):
    ✓ Revisar documentación del esquema antes de consultar
    ✓ Usar herramientas de exploración (DBeaver, pgAdmin)
    ✓ Realizar LIMIT 10 para inspeccionar antes
    ✓ Validar que tabla esté actualizada en entornos reales
    ✓ Usar alias para evitar ambigüedad con múltiples tablas
    
    FILTRADO DE RESULTADOS (WHERE):
    ✓ Filtrar lo antes posible para reducir carga
    ✓ Validar tipos de datos (números sin comillas)
    ✓ Indexar campos filtrados frecuentemente
    ✓ Documentar condiciones para trazabilidad
    ✓ Combinar condiciones con AND/OR logicamente
    
    ═════════════════════════════════════════════════════════════
    ERRORES COMUNES A EVITAR:
    ═════════════════════════════════════════════════════════════
    
    ✗ SELECT *: Carga innecesaria de datos
    ✗ WHERE total = '1000': Comillas en números
    ✗ Omitir WHERE: Consulta demasiado amplia
    ✗ LIKE ineficiente: %texto% consume recursos
    ✗ No indexar WHERE: Queries lentas en datos grandes
    ✗ Confundir tablas: Resultados incorrectos
"""

print(buenas_practicas)

## 7. RESUMEN (Slide 35)

In [ ]:
resumen = """
    SLIDE 35: RESUMEN DE SESIÓN 4
    ═════════════════════════════════════════════════════════════
    
    1️⃣  SELECT
        → Especificar columnas necesarias
        → Usar alias AS para claridad
        → Evitar * en producción
    
    2️⃣  FROM
        → Identificar tabla correcta
        → Usar herramientas de exploración
        → Aplicar alias con múltiples tablas
    
    3️⃣  WHERE
        → Filtrar con condiciones precisas
        → Combinar AND/OR/NOT logicamente
        → Usar IN, BETWEEN, LIKE según necesidad
    
    4️⃣  BUENAS PRÁCTICAS
        → Filtrar tempranamente en la query
        → Validar tipos de datos
        → Documentar condiciones
    
    5️⃣  APLICACIÓN PRÁCTICA
        → Resolver consultas logísticas reales
        → Generar reportes filtrados
        → Integrar con dashboards
    
    ═════════════════════════════════════════════════════════════
    SIGUIENTE SESIÓN:
    
    Operadores y condiciones avanzadas:
    • IN, NOT IN, BETWEEN
    • AND, OR, operadores lógicos
    • Consultas más complejas y flexibles
"""

print(resumen)